# Part A: GAN Improvements and Experiments

One sentence stating the goal: starting from the trained baseline DCGAN, isolate which changes help, then combine the winners into a single final model.

## 1. Imports and Setup

In [ ]:
import json
import os
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from torchvision.transforms import transforms
import torch.nn.functional as F
from torchvision.transforms import functional as F_t
from torchvision.datasets import CIFAR10
import datasets 
import matplotlib.pyplot as plt

# Match the baseline's random seed and use the GPU when one is available.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
print(f'running on {device}')
cpu_cores = min(16, os.cpu_count() or 1)


c:\Users\Admin\Desktop\st1504-ca2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


running on cuda


In [3]:
import seaborn as sns

# Shared design system for all notebooks in dele/: a 10-color categorical palette
# (one hue per CIFAR10 class), validated for lightness, chroma, and colorblind
# separation with the dataviz skill's validator (all checks pass).
PALETTE = [
    "#A1533F",  # 0 airplane   - terracotta
    "#C08A3E",  # 1 automobile - ochre
    "#3D8049",  # 2 bird       - sage
    "#24558F",  # 3 cat        - slate blue
    "#8A3466",  # 4 deer       - plum
    "#4552A8",  # 5 dog        - indigo
    "#8C3F32",  # 6 frog       - brick red
    "#B79A2E",  # 7 horse      - mustard
    "#B5615B",  # 8 ship       - rose
    "#6B7A1A",  # 9 truck      - olive
]
sns.set_theme(style="whitegrid", palette=PALETTE)
plt.rcParams["figure.facecolor"] = "#fcfcfb"
plt.rcParams["axes.facecolor"] = "#fcfcfb"

In [ ]:
class generator(nn.Module):
    def __init__(self, noise_dim):
        super().__init__()
        self.proj = nn.Linear(noise_dim + 10, 128*4*4)
        self.bn_proj = nn.BatchNorm1d(128*4*4)
        self.convT1 = nn.ConvTranspose2d(128, 64, 4, 2, 1) # 4x4 -> 8x8
        self.bn1 = nn.BatchNorm2d(64)

        self.convT2 = nn.ConvTranspose2d(64, 32, 4, 2, 1) # 8x8 -> 16x16
        self.bn2 = nn.BatchNorm2d(32)

        self.convT3 = nn.ConvTranspose2d(32, 3, 4, 2, 1) # 16x16 -> 32x32

        #activations
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()



    def forward(self, noise, label):
        x = torch.concat((noise, label), dim=-1)
        x = self.relu(self.bn_proj(self.proj(x)))
        x = x.reshape(-1, 128, 4, 4)
        
        x = self.relu(self.bn1(self.convT1(x)))
        x = self.relu(self.bn2(self.convT2(x)))
        x = self.tanh(self.convT3(x))
        return x

class discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(13, 32, 3, 2, 1) # 32x32 -> 16x16

        self.conv2 = nn.Conv2d(32, 64, 3, 2, 1) # 16x16 -> 8x8
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, 3, 2, 1) # 8x8 -> 4x4
        self.bn3 = nn.BatchNorm2d(128)

        self.fc_out = nn.Linear(128*4*4, 1) 
        # activations
        
        self.leakyrelu = nn.LeakyReLU(0.2)
        self.sigmoid = nn.Sigmoid()

        # reg
        self.dropout2d = nn.Dropout2d(0.3)
        
    def forward(self, x, label):
        batch_size = x.size(0)
        label = label.reshape(batch_size, 10, 1, 1).expand(batch_size, 10, 32, 32)
        x = torch.concat((x, label), dim=-3)
        x = self.leakyrelu(self.conv1(x))
        x = self.dropout2d(x)
        x = self.leakyrelu(self.bn2(self.conv2(x)))
        x = self.dropout2d(x)
        x = self.leakyrelu(self.bn3(self.conv3(x)))
        x = self.dropout2d(x)
        x = x.flatten(start_dim=1)
        x = self.sigmoid(self.fc_out(x))
        return x

noise_dim = 128
bs = 128
validation_n_samples=5
classes  = ['airplane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


fixed_noise_progression_grid = {}

D_x_track = {'train':[], 'val': []}
D_Gz_track = {'train':[], 'val': []}
real_d_loss_track = {'train':[], 'val': []}
fake_d_loss_track = {'train':[], 'val': []}
g_loss_track = {'train':[], 'val': []}

def set_requires_grad(model, yesno):
    for param in model.parameters():
        param.requires_grad_(yesno)

def print_template(D_x, D_Gz, real_d_loss, fake_d_loss, g_loss):
    print(f'D(x) = {D_x:.4f}')
    print(f'D(G(z)) = {D_Gz:.4f}')
    print(f'D Loss (real): {real_d_loss:.3f} | D Loss (fake): {fake_d_loss:.3f}')
    print(f'Generator Loss: {g_loss:.3f}')

val_noise = torch.randn((bs, noise_dim)).to(device)

def track_metrics(train_val, D_x, D_Gz, real_d_loss, fake_d_loss, g_loss):
    D_x_track[train_val].append(D_x.item())
    D_Gz_track[train_val].append(D_Gz.item())
    real_d_loss_track[train_val].append(real_d_loss.item())
    fake_d_loss_track[train_val].append(fake_d_loss.item())
    g_loss_track[train_val].append(g_loss.item())


def generate_class_images(G, n, e):
    G.eval()
    fixed_noise_progression_grid[e + 1] = []

    with torch.no_grad():
        fig, ax = plt.subplots(n, 10, figsize=(12, 5))
        for col, a in enumerate(ax[0]):
            a.set_title(f"{classes[col]}", pad=10)

        for idx in range(10):
            one_hot_label = [0] * 10
            one_hot_label[idx] = 1
            one_hot_label = torch.Tensor(one_hot_label).expand(n, -1).to(device)
            noise = val_noise[0:n].to(device)
            per_class_generated_imgs = G(noise, one_hot_label)

            fixed_noise_progression_grid[e + 1].append(
                per_class_generated_imgs[0].detach().cpu().numpy()
            )

            for img in range(per_class_generated_imgs.size(0)):
                ax[img][idx].imshow(F_t.to_pil_image(
                    (127.5 * (per_class_generated_imgs[img] + 1)).to(torch.uint8)
                ))
                ax[img][idx].axis("off")

        plt.tight_layout()
        plt.show()



## 2. Load Baseline Results

The saved baseline configuration, final losses, eye-test scores, and FID are loaded without retraining, giving every experiment the same fixed reference point.

In [ ]:
def one_hot_encode(batch):
    labels = []
    for n in batch:
        r = [0]*10
        r[n.item()] = 1
        labels.append(r)
    return {'label': labels}
image_data = datasets.load_dataset('uoft-cs/cifar10').with_format('torch')
train_data = image_data['train'].map(lambda x: {'img': x/127.5 - 1}, input_columns=['img'], batched=True, batch_size=500, num_proc=cpu_cores)
test_data = image_data['test'].map(lambda x: {'img': x/127.5 - 1}, input_columns=['img'], batched=True, batch_size=500, num_proc=cpu_cores)
sample_batch = torch.tensor(test_data.select(range(64)).with_format(None)['img'])
print(f'range: {sample_batch.min()} to {sample_batch.max()}')

train_data = train_data.map(one_hot_encode, input_columns=['label'], batched=True, batch_size=500, num_proc=cpu_cores)
test_data = test_data.map(one_hot_encode, input_columns=['label'], batched=True, batch_size=500, num_proc=cpu_cores)

val_data = train_data.shuffle(seed=42).select(range(5000))
train_data = train_data.shuffle(seed=42).select(range(45000))


range: -1.0 to 1.0


In [ ]:
# shit ignore this 



# BASELINE_RESULTS_PATH = Path("gan_baseline_results.json")
# if not BASELINE_RESULTS_PATH.exists():
#     raise FileNotFoundError(
#         "gan_baseline_results.json was not found. Run gan_baseline.ipynb "
#         "through its result-saving cell before running this notebook."
#     )

# with BASELINE_RESULTS_PATH.open(encoding="utf-8") as file:
#     baseline_results = json.load(file)

# baseline_eye_scores = baseline_results["eye_test_scores"]


# baseline_class_quality = {
#     class_name: float(np.mean(baseline_eye_scores[class_name]))
#     for class_name in classes
# }
# baseline_quality_score = float(np.mean(list(baseline_class_quality.values())))
# baseline_loss_balance = abs(
#     baseline_results["final_val_g_loss"] - baseline_results["final_val_d_loss"]
# )

# baseline_summary = pd.Series(
#     {
#         "quality_score": baseline_quality_score,
#         "final_val_g_loss": baseline_results["final_val_g_loss"],
#         "final_val_d_loss": baseline_results["final_val_d_loss"],
#         "g_d_loss_gap": baseline_loss_balance,
#         "fid_score": baseline_results["fid_score"],
#     },
#     name="baseline",
# ).to_frame("value")
# baseline_summary

In [8]:
BASELINE_RESULTS_PATH = Path("gan_baseline_results.json")
if not BASELINE_RESULTS_PATH.exists():
    raise FileNotFoundError(
        "gan_baseline_results.json was not found. Run gan_baseline.ipynb "
        "through its result-saving cell before running this notebook."
    )

with BASELINE_RESULTS_PATH.open(encoding="utf-8") as file:
    baseline_results = json.load(file)

baseline_eye_scores = baseline_results["eye_test_scores"] # this is such a stupid metric but its required in the assignment apparently -_-

baseline_class_quality = {
    class_name: float(np.mean(baseline_eye_scores[class_name]))
    for class_name in classes
}
baseline_quality_score = float(np.mean(list(baseline_class_quality.values())))

baseline_summary = pd.Series(
    {
        "quality_score": baseline_quality_score,
        "final_val_g_loss": baseline_results["final_val_g_loss"],
        "final_val_real_d_loss": baseline_results["final_val_d_loss"],
        "final_val_fake_d_loss": 0, # fake value ill fill this up once i rerun baseline
        "fid_score": baseline_results["fid_score"],
    },
    name="baseline",
).to_frame("value")
baseline_summary

,value
quality_score,0.225000
final_val_g_loss,0.905743
final_val_real_d_loss,1.117043
final_val_fake_d_loss,0.000000
fid_score,114.150101


## 3. Comparison Metrics

One sentence stating the goal: define, before running any experiment, exactly how "better" will be measured and how the best setup will be picked, so results aren't judged after the fact by eyeballing.

### 3.1 Quality Score (primary metric)

One sentence defining the quality score: convert each image's eye-test label to a number (clear=1, marginal=0.5, nonsense=0) and average across the scored sample, per class and overall — the identical definition used in `vae_improvement.ipynb`, so GAN and VAE results stay comparable.

In [ ]:
# Define score_from_labels(): map a list of clear/marginal/nonsense tallies to a 0-1 quality score, reused by every experiment.
def score_from_labels(labels):
    # Marginal counts as half-credit rather than being dropped, since it's a real (if weaker) signal of quality, not noise.
    weights = {"c": 1.0, "m": 0.5, "n": 0.0}
    return sum(weights[label] for label in labels) / len(labels)


### 3.2 Discriminator/Generator Loss Balance (secondary / diagnostic metric)

One sentence defining the secondary metric: the final gap between generator and discriminator loss, tracked per model as a cheap automatic check for collapse or one network overpowering the other — the GAN analogue of the VAE's validation loss, since GANs have no single validation loss to rely on. FID (loaded from the baseline's JSON, recomputed only for the Section 10 final model) is tracked alongside these as a non-self-graded quantitative check, not used to rank experiments 1-6.

### 3.3 Decision Rule

One sentence stating the rule used to pick a winner: quality score is the primary ranking metric since it's what the assignment actually grades (image quality); the G/D loss balance is checked only as a tiebreaker or red flag (e.g. reject a high-quality-score model if training clearly collapsed) — this rule is fixed here, before any results exist, so it can't be quietly bent to favor a preferred outcome.

In [32]:
# Initialize a shared results dict (starting with the baseline's numbers) that every experiment appends its quality score and loss balance to.
# Note: experiments 1-6 log metrics only and do NOT save .h5 weights — only the baseline and the Section 10 final model get saved weights.
results = {
    "baseline": baseline_summary.to_dict()['value']
}
results = pd.DataFrame(results).T
results

,quality_score,final_val_g_loss,final_val_real_d_loss,final_val_fake_d_loss,fid_score
baseline,0.225,0.905743,1.117043,0.0,114.150101


In [51]:
def add_row_to_df(df, row, experiment_name):
    return pd.concat([df, pd.DataFrame({experiment_name:row}).T], ignore_index=False)
add_row_to_df(results, baseline_summary.to_dict()['value'], 'hello 67')


,quality_score,final_val_g_loss,final_val_real_d_loss,final_val_fake_d_loss,fid_score
baseline,0.225,0.905743,1.117043,0.0,114.150101
hello 67,0.225,0.905743,1.117043,0.0,114.150101


## 4. Experiment 1: TTUR (Two Time-Scale Update Rule)

Hypothesis: the baseline uses one shared learning rate for both networks, which risks the discriminator learning faster than the generator and overpowering it; giving the discriminator a lower learning rate than the generator should raise the quality score by keeping their contest balanced for longer — checked against the risk of slowing convergence too much if the ratio is too extreme.

In [ ]:
# Build and train a GAN identical to baseline except for separate, unequal generator/discriminator learning rates (TTUR).

criterion = nn.BCELoss()

train_loader = DataLoader(train_data,batch_size=bs,shuffle=True,
                          num_workers=cpu_cores,pin_memory=True,persistent_workers=True,prefetch_factor=2)
val_loader = DataLoader(val_data,batch_size=bs,shuffle=True,drop_last=True,
                        num_workers=cpu_cores,pin_memory=True,persistent_workers=True,prefetch_factor=2)

fixed_noise_progression_grid = {}

D_x_track = {'train':[], 'val': []}
D_Gz_track = {'train':[], 'val': []}
real_d_loss_track = {'train':[], 'val': []}
fake_d_loss_track = {'train':[], 'val': []}
g_loss_track = {'train':[], 'val': []}

In [ ]:
G = generator(noise_dim).to(device)
D = discriminator().to(device)
g_lr = 0.0003
d_lr = 0.0001
g_optim = torch.optim.Adam(G.parameters(), lr=g_lr, betas=(0.5, 0.9))
d_optim = torch.optim.Adam(D.parameters(), lr=d_lr, betas=(0.5, 0.9))
epochs = 50

fixed_noise_progression_grid = {}

D_x_track = {'train':[], 'val': []}
D_Gz_track = {'train':[], 'val': []}
real_d_loss_track = {'train':[], 'val': []}
fake_d_loss_track = {'train':[], 'val': []}
g_loss_track = {'train':[], 'val': []}

In [ ]:
for e in range(epochs):
    D.train()
    G.train()
    print(f'\n\nEpoch {e}\n')
    for idx, sample in enumerate(train_loader):
   
        img = sample['img'].to(device, non_blocking=True)
        label = sample['label'].to(device, non_blocking=True)

        batch_size = img.size(0)

        target_ones = torch.ones(batch_size).unsqueeze(1).to(device)
        target_zeros = torch.zeros(batch_size).unsqueeze(1).to(device)
        
        # generate fake image batch
        noise = torch.randn((batch_size,noise_dim)).to(device)
        generated_img = G(noise, label)
        # unfreeze discriminator 
        # reset discriminator optimizer accumulated gradients
        set_requires_grad(D, True)
        d_optim.zero_grad() 

        real_d_pred = D(img, label)
        real_d_loss = criterion(real_d_pred, target_ones) # predict D(x) = 1

        fake_d_pred = D(generated_img.detach(), label)
        fake_d_loss = criterion(fake_d_pred, target_zeros) # predict D(G(z)) = 0

        # d_loss = loss(real) + loss(fake)
        # backprop -> step
        d_loss = real_d_loss + fake_d_loss
        d_loss.backward()
        d_optim.step()

        # freeze discriminator
        set_requires_grad(D, False)
        # reset generator optimizer accumulated gradients
        g_optim.zero_grad()

        g_pred = D(generated_img, label) # uses the new discriminator
        g_loss = criterion(g_pred, target_ones) # predict D(G(z)) = 1
        g_loss.backward()
        g_optim.step()

        # repredicting so that this uses the updated discriminator for a fairer comparison
        with torch.no_grad():
            real_d_pred = D(img, label)
        
        D_x = real_d_pred.mean()
        D_Gz = g_pred.mean()
    print_template(D_x, D_Gz, real_d_loss, fake_d_loss, g_loss)
    track_metrics('train', D_x, D_Gz, real_d_loss, fake_d_loss, g_loss)
    # validation
    D.eval()
    G.eval()
    for idx, v_sample in enumerate(val_loader):
        
        img = v_sample['img'].to(device, non_blocking=True)
        label = v_sample['label'].to(device, non_blocking=True)

        batch_size = img.size(0) # lol

        target_ones = torch.ones(batch_size).unsqueeze(1).to(device)
        target_zeros = torch.zeros(batch_size).unsqueeze(1).to(device)
        with torch.no_grad():
            v_generated_img = G(val_noise, label)

            v_real_pred = D(img, label) 
                        
            v_fake_pred = D(v_generated_img, label)

            v_real_d_loss = criterion(v_real_pred, target_ones)
            v_fake_d_loss = criterion(v_fake_pred, target_zeros)
            v_g_loss = criterion(v_fake_pred, target_ones)

            v_D_x = v_real_pred.mean()
            v_D_Gz = v_fake_pred.mean()
    track_metrics('val', v_D_x, v_D_Gz, v_real_d_loss, v_fake_d_loss, v_g_loss)
    print('== Validation ==')
    print_template(v_D_x, v_D_Gz, v_real_d_loss, v_fake_d_loss, v_g_loss)
    # if (e+1)%5 == 0:
    #     generate_class_images(G, validation_n_samples, e)


In [ ]:
# Generate a sample of images, score with score_from_labels() (Section 3.1), and log quality score + final G/D loss gap into the results dict.

In [ ]:
# Plot this experiment's generator/discriminator loss curves against the baseline's, on the same axes.

### Notes

One sentence stating whether TTUR raised the quality score over the baseline and whether the G/D loss curves look more balanced than baseline's.

## 5. Experiment 2: Label Smoothing

Hypothesis: the baseline's hard 0/1 discriminator targets can make it overconfident and easy to saturate, so one-sided label smoothing (real target 0.9 instead of 1.0) should raise the quality score by keeping discriminator gradients informative for longer — checked against the risk of smoothing too much and weakening the discriminator's signal to the generator.

In [ ]:
G = generator(noise_dim).to(device)
D = discriminator().to(device)
g_lr = 0.0002
d_lr = 0.0002
g_optim = torch.optim.Adam(G.parameters(), lr=g_lr, betas=(0.5, 0.9))
d_optim = torch.optim.Adam(D.parameters(), lr=d_lr, betas=(0.5, 0.9))
epochs = 50

fixed_noise_progression_grid = {}

D_x_track = {'train':[], 'val': []}
D_Gz_track = {'train':[], 'val': []}
real_d_loss_track = {'train':[], 'val': []}
fake_d_loss_track = {'train':[], 'val': []}
g_loss_track = {'train':[], 'val': []}

In [ ]:
# Build and train a GAN identical to baseline except the discriminator's real-image target is smoothed from 1.0 to 0.9.
for e in range(epochs):
    D.train()
    G.train()
    print(f'\n\nEpoch {e}\n')
    for idx, sample in enumerate(train_loader):
   
        img = sample['img'].to(device, non_blocking=True)
        label = sample['label'].to(device, non_blocking=True)

        batch_size = img.size(0)

        target_ones = torch.ones(batch_size).unsqueeze(1).to(device)
        target_smooth = torch.full(batch_size, 0.9).unsqueeze(1).to(device)
        target_zeros = torch.zeros(batch_size).unsqueeze(1).to(device)
        
        # generate fake image batch
        noise = torch.randn((batch_size,noise_dim)).to(device)
        generated_img = G(noise, label)
        # unfreeze discriminator 
        # reset discriminator optimizer accumulated gradients
        set_requires_grad(D, True)
        d_optim.zero_grad() 

        real_d_pred = D(img, label)
        real_d_loss = criterion(real_d_pred, target_smooth) # predict D(x) = 1

        fake_d_pred = D(generated_img.detach(), label)
        fake_d_loss = criterion(fake_d_pred, target_zeros) # predict D(G(z)) = 0

        # d_loss = loss(real) + loss(fake)
        # backprop -> step
        d_loss = real_d_loss + fake_d_loss
        d_loss.backward()
        d_optim.step()

        # freeze discriminator
        set_requires_grad(D, False)
        # reset generator optimizer accumulated gradients
        g_optim.zero_grad()

        g_pred = D(generated_img, label) # uses the new discriminator
        g_loss = criterion(g_pred, target_ones) # predict D(G(z)) = 1
        g_loss.backward()
        g_optim.step()

        # repredicting so that this uses the updated discriminator for a fairer comparison
        with torch.no_grad():
            real_d_pred = D(img, label)
        
        D_x = real_d_pred.mean()
        D_Gz = g_pred.mean()
    print_template(D_x, D_Gz, real_d_loss, fake_d_loss, g_loss)
    track_metrics('train', D_x, D_Gz, real_d_loss, fake_d_loss, g_loss)
    # validation
    D.eval()
    G.eval()
    for idx, v_sample in enumerate(val_loader):
        
        img = v_sample['img'].to(device, non_blocking=True)
        label = v_sample['label'].to(device, non_blocking=True)

        batch_size = img.size(0) # lol

        target_ones = torch.ones(batch_size).unsqueeze(1).to(device)
        target_zeros = torch.zeros(batch_size).unsqueeze(1).to(device)
        with torch.no_grad():
            v_generated_img = G(val_noise, label)

            v_real_pred = D(img, label) 
                        
            v_fake_pred = D(v_generated_img, label)

            v_real_d_loss = criterion(v_real_pred, target_ones)
            v_fake_d_loss = criterion(v_fake_pred, target_zeros)
            v_g_loss = criterion(v_fake_pred, target_ones)

            v_D_x = v_real_pred.mean()
            v_D_Gz = v_fake_pred.mean()
    track_metrics('val', v_D_x, v_D_Gz, v_real_d_loss, v_fake_d_loss, v_g_loss)
    print('== Validation ==')
    print_template(v_D_x, v_D_Gz, v_real_d_loss, v_fake_d_loss, v_g_loss)
    # if (e+1)%5 == 0:
    #     generate_class_images(G, validation_n_samples, e)


In [ ]:
# Generate a sample of images, score with score_from_labels() (Section 3.1), and log quality score + final G/D loss gap into the results dict.

### Notes

One sentence stating whether label smoothing raised the quality score over the baseline and whether the G/D loss curves look more balanced than baseline's.

In [ ]:
# Plot this experiment's generator/discriminator loss curves against the baseline's, on the same axes.

## 6. Experiment 3: Spectral Normalization

Hypothesis: the baseline discriminator's weights are unconstrained and can grow to dominate the generator, so applying spectral normalization to the discriminator's conv layers should raise the quality score by capping its Lipschitz constant and stabilizing training — checked against the risk of over-constraining the discriminator so it can no longer distinguish real from fake effectively.

In [ ]:
# Build and train a GAN identical to baseline except the discriminator's conv layers are wrapped with spectral normalization.
from torch.nn.utils.parametrizations import spectral_norm

class spectral_discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = spectral_norm(nn.Conv2d(13, 32, 3, 2, 1)) # 32x32 -> 16x16

        self.conv2 = spectral_norm(nn.Conv2d(32, 64, 3, 2, 1)) # 16x16 -> 8x8

        self.conv3 = spectral_norm(nn.Conv2d(64, 128, 3, 2, 1)) # 8x8 -> 4x4

        self.fc_out = spectral_norm(nn.Linear(128*4*4, 1)) 
        # activations
        
        self.leakyrelu = nn.LeakyReLU(0.2)
        self.sigmoid = nn.Sigmoid()

        # reg
        self.dropout2d = nn.Dropout2d(0.3)
        
    def forward(self, x, label):
        batch_size = x.size(0)
        label = label.reshape(batch_size, 10, 1, 1).expand(batch_size, 10, 32, 32)
        x = torch.concat((x, label), dim=-3)
        x = self.leakyrelu(self.conv1(x))
        x = self.dropout2d(x)
        x = self.leakyrelu(self.bn2(self.conv2(x)))
        x = self.dropout2d(x)
        x = self.leakyrelu(self.bn3(self.conv3(x)))
        x = self.dropout2d(x)
        x = x.flatten(start_dim=1)
        x = self.sigmoid(self.fc_out(x))
        return x

In [ ]:
G = generator(noise_dim).to(device)
D = spectral_discriminator().to(device)

g_lr = 0.0002
d_lr = 0.0002
g_optim = torch.optim.Adam(G.parameters(), lr=g_lr, betas=(0.5, 0.9))
d_optim = torch.optim.Adam(D.parameters(), lr=d_lr, betas=(0.5, 0.9))
epochs = 50

fixed_noise_progression_grid = {}

D_x_track = {'train':[], 'val': []}
D_Gz_track = {'train':[], 'val': []}
real_d_loss_track = {'train':[], 'val': []}
fake_d_loss_track = {'train':[], 'val': []}
g_loss_track = {'train':[], 'val': []}

In [ ]:
# Build and train a GAN identical to baseline except the discriminator's real-image target is smoothed from 1.0 to 0.9.
for e in range(epochs):
    D.train()
    G.train()
    print(f'\n\nEpoch {e}\n')
    for idx, sample in enumerate(train_loader):
   
        img = sample['img'].to(device, non_blocking=True)
        label = sample['label'].to(device, non_blocking=True)

        batch_size = img.size(0)

        target_ones = torch.ones(batch_size).unsqueeze(1).to(device)
        target_zeros = torch.zeros(batch_size).unsqueeze(1).to(device)
        
        # generate fake image batch
        noise = torch.randn((batch_size,noise_dim)).to(device)
        generated_img = G(noise, label)
        # unfreeze discriminator 
        # reset discriminator optimizer accumulated gradients
        set_requires_grad(D, True)
        d_optim.zero_grad() 

        real_d_pred = D(img, label)
        real_d_loss = criterion(real_d_pred, target_ones) # predict D(x) = 1

        fake_d_pred = D(generated_img.detach(), label)
        fake_d_loss = criterion(fake_d_pred, target_zeros) # predict D(G(z)) = 0

        # d_loss = loss(real) + loss(fake)
        # backprop -> step
        d_loss = real_d_loss + fake_d_loss
        d_loss.backward()
        d_optim.step()

        # freeze discriminator
        set_requires_grad(D, False)
        # reset generator optimizer accumulated gradients
        g_optim.zero_grad()

        g_pred = D(generated_img, label) # uses the new discriminator
        g_loss = criterion(g_pred, target_ones) # predict D(G(z)) = 1
        g_loss.backward()
        g_optim.step()

        # repredicting so that this uses the updated discriminator for a fairer comparison
        with torch.no_grad():
            real_d_pred = D(img, label)
        
        D_x = real_d_pred.mean()
        D_Gz = g_pred.mean()
    print_template(D_x, D_Gz, real_d_loss, fake_d_loss, g_loss)
    track_metrics('train', D_x, D_Gz, real_d_loss, fake_d_loss, g_loss)
    # validation
    D.eval()
    G.eval()
    for idx, v_sample in enumerate(val_loader):
        
        img = v_sample['img'].to(device, non_blocking=True)
        label = v_sample['label'].to(device, non_blocking=True)

        batch_size = img.size(0) # lol

        target_ones = torch.ones(batch_size).unsqueeze(1).to(device)
        target_zeros = torch.zeros(batch_size).unsqueeze(1).to(device)
        with torch.no_grad():
            v_generated_img = G(val_noise, label)

            v_real_pred = D(img, label) 
                        
            v_fake_pred = D(v_generated_img, label)

            v_real_d_loss = criterion(v_real_pred, target_ones)
            v_fake_d_loss = criterion(v_fake_pred, target_zeros)
            v_g_loss = criterion(v_fake_pred, target_ones)

            v_D_x = v_real_pred.mean()
            v_D_Gz = v_fake_pred.mean()
    track_metrics('val', v_D_x, v_D_Gz, v_real_d_loss, v_fake_d_loss, v_g_loss)
    print('== Validation ==')
    print_template(v_D_x, v_D_Gz, v_real_d_loss, v_fake_d_loss, v_g_loss)
    # if (e+1)%5 == 0:
    #     generate_class_images(G, validation_n_samples, e)


### Notes

One sentence stating whether spectral normalization raised the quality score over the baseline and whether the G/D loss curves look more stable than baseline's.

In [ ]:
# Generate a sample of images, score with score_from_labels() (Section 3.1), and log quality score + final G/D loss gap into the results dict.

In [ ]:
# Plot this experiment's generator/discriminator loss curves against the baseline's, on the same axes.

## 7. Experiment 4: Color vs. Grayscale

Hypothesis (linked to the assignment's discussion question): converting inputs to grayscale removes color as a distinguishing cue, so the quality score should drop for classes that rely heavily on color but may hold up or even simplify shape-dominant classes — this experiment answers the question empirically rather than by reasoning alone.

In [ ]:
# Convert the training images to grayscale (single channel) and adjust the generator/discriminator's channel counts accordingly.

In [ ]:
# Build and train a GAN identical to baseline except for the single-channel input/output.

In [ ]:
# Generate a sample of images, score with score_from_labels() (Section 3.1) split by class, and log quality score + loss balance into the results dict.

In [ ]:
# Plot this experiment's generator/discriminator loss curves against the color baseline's, on the same axes.

### Discussion: Color vs. Grayscale, Empirically

One sentence stating whether this experiment's quality score confirmed or contradicted the prediction made in `gan_baseline.ipynb` Section 10.4, with the per-class score breakdown as evidence.

## 8. Experiment 5: Engineered Conditioning Features

Hypothesis: a bare one-hot label only tells the generator/discriminator "which of 10 buckets," so concatenating each class's mean per-channel color statistics (from `vae_eda.ipynb` Section 5, the same shared EDA both architectures use) alongside the one-hot vector should raise the quality score by giving both networks a richer conditioning signal — checked against the risk that the auxiliary features are redundant with one-hot or add noise the discriminator has to learn to ignore.

In [ ]:
# Compute each class's mean per-channel (R, G, B) statistics from the training set, building a small per-class auxiliary feature vector.

In [ ]:
# Build and train a GAN identical to baseline except the conditioning input is [one-hot label, auxiliary color-stats vector] instead of one-hot alone.

In [ ]:
# Generate a sample of images, score with score_from_labels() (Section 3.1), and log quality score + final G/D loss gap into the results dict.

In [ ]:
# Plot this experiment's generator/discriminator loss curves against the baseline's, on the same axes.

### Notes

One sentence stating whether the engineered conditioning features raised the quality score over the baseline's plain one-hot conditioning.

## 9. Experiment 6: Data Augmentation

Hypothesis: augmentation (e.g. random horizontal flip) exposes the discriminator to more visual variation per class without new data, which should raise the quality score by making the discriminator generalize instead of memorizing exact training images (a stronger discriminator gives the generator a better training signal) — checked against the risk that some flips are semantically wrong for a class and could confuse conditioning rather than help it.

In [ ]:
# Define an augmentation pipeline (e.g. random horizontal flip) applied to the training set only.

In [ ]:
# Build and train a GAN identical to baseline except the discriminator's real-image inputs are passed through the augmentation pipeline.

In [ ]:
# Generate a sample of images, score with score_from_labels() (Section 3.1), and log quality score + final G/D loss gap into the results dict.

In [ ]:
# Plot this experiment's generator/discriminator loss curves against the baseline's, on the same axes.

### Notes

One sentence stating whether augmentation raised the quality score over the baseline and whether the discriminator's training signal looks stronger as a result.

## 10. Final Model

One sentence stating the goal: apply the decision rule from Section 3.3 to combine the winning settings from experiments 1-3, 5, and 6 (TTUR, label smoothing, spectral normalization, engineered conditioning, augmentation) into the single final model — Experiment 4 (color vs. grayscale) targets a discussion question, not a quality lever, so it's evaluated separately and not folded into the combination. This is the only model besides the baseline whose weights get saved.

### 10.1 Select Best Settings

In [ ]:
# Read each experiment's quality score/loss balance from the results dict, and pick the best setting per axis using the Section 3.3 decision rule.

### 10.2 Train the Final Model

In [ ]:
# Build and train a GAN using the combined best settings from 10.1 (no weights saved yet — training only).

In [ ]:
# Save the final model's generator weights to .h5 — the deliverable weights file for this notebook, alongside the baseline's.

### 10.3 Evaluate the Final Model

In [ ]:
# Generate 1000 images (100/class) with the final model, score with score_from_labels(), and log into the results dict against the baseline.

In [ ]:
# Compute FID between real validation images and the final model's 1000 generated images (same InceptionV3 extractor as baseline), log into the results dict.

In [ ]:
# Plot the final model's generator/discriminator loss curves against the baseline's, on the same axes.

### Notes

One sentence naming the final model's quality score and FID against the baseline's, stating the size of the improvement (or lack of one).

## 11. Ablation Summary

One sentence noting this table pulls the shared results dict into one place, so the final model's choice is backed by numbers already computed during the experiments, not a new comparison step.

In [ ]:
# Build one ablation table from the results dict: baseline, each experiment, and the final model, with quality score, G/D loss balance, and FID (baseline/final only) as columns.

In [ ]:
# Bar chart comparing quality scores across all models.

## 12. Conclusion

One sentence summarizing which single changes raised the quality score and which didn't (per the Section 3 metrics), how much the final model gained over the baseline, and noting its weights are saved to .h5 alongside the baseline's as the Part A GAN deliverable, feeding into `vae_gan_comparison.ipynb`.